# 🛠️ Template: Feature Engineering

Plantilla reutilizable para creación y transformación de variables.  
Organizada de **básico a avanzado** — usá solo las secciones que necesitás.

---

## ¿Cómo usar esta plantilla?

1. Copiá las celdas de las técnicas que necesitás
2. Cambiá las variables marcadas con `# ← CAMBIAR`
3. Siempre trabajá sobre una copia del dataset limpio (`df_clean`)

**Orden recomendado en un proyecto real:**
```
Limpieza → Feature Engineering → Escalado → Encoding → Modelado
```

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, f_regression

# ── Configuración ──────────────────────────────────────────
# Siempre trabajar sobre una copia para preservar el dataset limpio original
# df_clean = df.copy()  # ← descomentar si aún no tenés df_clean
# ───────────────────────────────────────────────────────────

---
# BLOQUE 1 — Creación de Variables Derivadas (Básico)

Crear nuevas columnas a partir de operaciones simples entre columnas existentes.  
**Cuándo usarlo:** cuando la combinación de dos variables tiene más poder predictivo que cada una por separado.

In [ ]:
# ── Configuración ──────────────────────────────────────────
COL_A = 'columna_1'  # ← CAMBIAR
COL_B = 'columna_2'  # ← CAMBIAR
# ───────────────────────────────────────────────────────────

# Operaciones aritméticas básicas
df_clean['diferencia']    = df_clean[COL_A] - df_clean[COL_B]
df_clean['suma']          = df_clean[COL_A] + df_clean[COL_B]
df_clean['ratio']         = df_clean[COL_A] / df_clean[COL_B].replace(0, np.nan)  # evita división por cero
df_clean['producto']      = df_clean[COL_A] * df_clean[COL_B]

print('✅ Variables derivadas creadas')
display(df_clean[[COL_A, COL_B, 'diferencia', 'suma', 'ratio', 'producto']].head())

---
# BLOQUE 2 — Reglas de Negocio con .apply() (Básico)

Crear columnas a partir de lógica condicional compleja definida como función.  
**Cuándo usarlo:** cuando una nueva variable depende de condiciones sobre múltiples columnas.

In [ ]:
# ── Configuración ──────────────────────────────────────────
NUEVA_COLUMNA = 'nueva_variable'  # ← CAMBIAR: nombre de la columna resultante
# ───────────────────────────────────────────────────────────

def regla_de_negocio(fila):
    """
    Definí acá tu lógica. Tiene acceso a todas las columnas de la fila.
    Ejemplo: calcular un costo estimado según condiciones.
    """
    # ── Ejemplo: costo mensual estimado ───────────────────
    if fila['columna_condicion'] == 'No':   # ← CAMBIAR
        return 0
    else:
        base  = 20                          # ← CAMBIAR
        extra = (fila['columna_numerica'] - 1) * 10  # ← CAMBIAR
        return base + extra

# axis=1 indica que la función se aplica fila por fila
df_clean[NUEVA_COLUMNA] = df_clean.apply(regla_de_negocio, axis=1)

print(f'✅ Columna "{NUEVA_COLUMNA}" creada')
print(df_clean[NUEVA_COLUMNA].describe())

---
# BLOQUE 3 — Binning / Discretización (Básico-Intermedio)

Convertir una variable numérica continua en categorías (rangos).  
**Cuándo usarlo:** cuando querés segmentar (ej: edad en joven/adulto/mayor, ingreso en bajo/medio/alto).

In [ ]:
# ── Configuración ──────────────────────────────────────────
COL_NUMERICA  = 'columna_numerica'    # ← CAMBIAR
NUEVA_COL_BIN = 'columna_categoria'   # ← CAMBIAR
BINS   = [0, 25, 50, 75, 100]         # ← CAMBIAR: límites de los rangos
LABELS = ['Bajo', 'Medio', 'Alto', 'Muy Alto']  # ← CAMBIAR: un label menos que bins
# ───────────────────────────────────────────────────────────

# Opción A: rangos manuales
df_clean[NUEVA_COL_BIN] = pd.cut(
    df_clean[COL_NUMERICA],
    bins=BINS,
    labels=LABELS,
    include_lowest=True
)

# Opción B: rangos automáticos por cuartiles (misma cantidad de datos en cada grupo)
# df_clean[NUEVA_COL_BIN] = pd.qcut(df_clean[COL_NUMERICA], q=4, labels=LABELS)

print(f'✅ Distribución de categorías en "{NUEVA_COL_BIN}":')
print(df_clean[NUEVA_COL_BIN].value_counts())

---
# BLOQUE 4 — Extracción de Fechas (Intermedio)

Descomponer una columna de fecha en sus componentes (año, mes, día, día de semana).  
**Cuándo usarlo:** cuando tenés una columna de fecha y necesitás analizar estacionalidad o tendencias.

In [ ]:
# ── Configuración ──────────────────────────────────────────
COL_FECHA = 'columna_fecha'  # ← CAMBIAR
# ───────────────────────────────────────────────────────────

# Asegurar que la columna es de tipo datetime
df_clean[COL_FECHA] = pd.to_datetime(df_clean[COL_FECHA])

# Extraer componentes
df_clean['año']              = df_clean[COL_FECHA].dt.year
df_clean['mes']              = df_clean[COL_FECHA].dt.month
df_clean['dia']              = df_clean[COL_FECHA].dt.day
df_clean['dia_semana']       = df_clean[COL_FECHA].dt.dayofweek   # 0=lunes, 6=domingo
df_clean['nombre_dia']       = df_clean[COL_FECHA].dt.day_name()
df_clean['trimestre']        = df_clean[COL_FECHA].dt.quarter
df_clean['es_fin_de_semana'] = df_clean[COL_FECHA].dt.dayofweek >= 5

# Antigüedad en días desde una fecha de referencia
df_clean['antiguedad_dias'] = (pd.Timestamp.today() - df_clean[COL_FECHA]).dt.days

print('✅ Componentes de fecha extraídos')
display(df_clean[[COL_FECHA, 'año', 'mes', 'dia', 'dia_semana', 'trimestre', 'es_fin_de_semana']].head())

---
# BLOQUE 5 — Encoding de Variables Categóricas (Intermedio)

Convertir texto en números para que los modelos de ML puedan procesarlos.  

| Técnica | Cuándo usarla |
|---|---|
| Label Encoding | Categorías con orden natural (bajo/medio/alto) |
| One-Hot Encoding | Categorías sin orden (ciudad, color, carrera) |
| Ordinal Encoding | Categorías con orden explícito definido por vos |

In [ ]:
# ── Configuración ──────────────────────────────────────────
COL_CATEGORICA     = 'columna_categorica'   # ← CAMBIAR
COLS_ONE_HOT       = ['col_1', 'col_2']     # ← CAMBIAR: columnas para One-Hot
# ───────────────────────────────────────────────────────────

# --- Opción A: Label Encoding (0, 1, 2, 3...) ---
le = LabelEncoder()
df_clean[f'{COL_CATEGORICA}_encoded'] = le.fit_transform(df_clean[COL_CATEGORICA])
print('Label Encoding:')
print(dict(zip(le.classes_, le.transform(le.classes_))))

# --- Opción B: One-Hot Encoding ---
# drop_first=True evita multicolinealidad (trampa de la variable ficticia)
df_clean = pd.get_dummies(df_clean, columns=COLS_ONE_HOT, drop_first=True)
print('\n✅ One-Hot Encoding aplicado')

# --- Opción C: Ordinal Encoding (orden explícito) ---
ORDEN = {'Bajo': 0, 'Medio': 1, 'Alto': 2}  # ← CAMBIAR
# df_clean[f'{COL_CATEGORICA}_ordinal'] = df_clean[COL_CATEGORICA].map(ORDEN)

print(f'Shape resultante: {df_clean.shape}')

---
# BLOQUE 6 — Transformaciones Matemáticas (Intermedio)

Transformar distribuciones sesgadas para que se aproximen a una distribución normal.  
**Cuándo usarlo:** antes de entrenar modelos que asumen normalidad (regresión lineal, Ridge, Lasso).

In [ ]:
# ── Configuración ──────────────────────────────────────────
COL_SESGO = 'columna_con_sesgo'  # ← CAMBIAR
# ───────────────────────────────────────────────────────────

# Transformación logarítmica (requiere valores positivos)
df_clean[f'{COL_SESGO}_log']   = np.log(df_clean[COL_SESGO].replace(0, np.nan))

# log1p: más segura, maneja ceros (log(1+x))
df_clean[f'{COL_SESGO}_log1p'] = np.log1p(df_clean[COL_SESGO])

# Raíz cuadrada (sesgo moderado)
df_clean[f'{COL_SESGO}_sqrt']  = np.sqrt(df_clean[COL_SESGO].clip(lower=0))

# Visualizar comparación de distribuciones
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df_clean[COL_SESGO].hist(ax=axes[0], bins=50); axes[0].set_title('Original')
df_clean[f'{COL_SESGO}_log1p'].hist(ax=axes[1], bins=50); axes[1].set_title('Log1p')
df_clean[f'{COL_SESGO}_sqrt'].hist(ax=axes[2], bins=50); axes[2].set_title('Raíz cuadrada')
plt.suptitle(f'Comparación de transformaciones — {COL_SESGO}')
plt.tight_layout()
plt.show()

---
# BLOQUE 7 — Escalado de Variables Numéricas (Intermedio)

Llevar todas las variables numéricas a una escala comparable.  

| Técnica | Cuándo usarla |
|---|---|
| StandardScaler | Modelos que asumen normalidad (regresión, SVM, redes neuronales) |
| MinMaxScaler | Cuando necesitás rango [0, 1] (KNN, redes neuronales) |

⚠️ **Importante:** hacer `fit` solo con datos de train para evitar data leakage.

In [ ]:
from sklearn.model_selection import train_test_split

# ── Configuración ──────────────────────────────────────────
COLUMNAS_ESCALAR = ['col_num_1', 'col_num_2', 'col_num_3']  # ← CAMBIAR
TARGET           = 'variable_objetivo'                       # ← CAMBIAR
TEST_SIZE        = 0.2
RANDOM_STATE     = 42
# ───────────────────────────────────────────────────────────

X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

# 1. Primero el split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# 2. Después el escalado — fit SOLO con train
scaler = StandardScaler()  # ← cambiar por MinMaxScaler() si lo necesitás
X_train[COLUMNAS_ESCALAR] = scaler.fit_transform(X_train[COLUMNAS_ESCALAR])
X_test[COLUMNAS_ESCALAR]  = scaler.transform(X_test[COLUMNAS_ESCALAR])  # solo transform

print(f'✅ Escalado aplicado')
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

---
# BLOQUE 8 — Selección de Features (Avanzado)

Identificar qué variables aportan más información al modelo y descartar las que agregan ruido.  
**Cuándo usarlo:** cuando tenés muchas columnas y querés reducir dimensionalidad sin PCA.

In [ ]:
# ── Configuración ──────────────────────────────────────────
K_FEATURES  = 10          # ← CAMBIAR: cuántas features mantener
TIPO_TARGET = 'regresion' # ← CAMBIAR: 'regresion' o 'clasificacion'
# ───────────────────────────────────────────────────────────

# Seleccionar función de scoring según tipo de problema
score_func = f_regression if TIPO_TARGET == 'regresion' else f_classif

selector = SelectKBest(score_func=score_func, k=K_FEATURES)
selector.fit(X_train, y_train)

# Ver importancia de cada feature
importancias = pd.DataFrame({
    'Feature': X_train.columns,
    'Score':   selector.scores_
}).sort_values('Score', ascending=False)

print(f'Top {K_FEATURES} features seleccionadas:')
display(importancias.head(K_FEATURES))

# Aplicar selección
cols_seleccionadas = importancias.head(K_FEATURES)['Feature'].tolist()
X_train_sel = X_train[cols_seleccionadas]
X_test_sel  = X_test[cols_seleccionadas]

print(f'\nShape con features seleccionadas: {X_train_sel.shape}')

---
# BLOQUE 9 — Reducción de Dimensionalidad con PCA (Avanzado)

Comprimir múltiples variables correlacionadas en componentes que resumen la información.  
**Cuándo usarlo:** cuando tenés muchas variables numéricas correlacionadas entre sí y querés reducirlas.  
⚠️ Las componentes de PCA no son interpretables directamente — sacrificás explicabilidad por rendimiento.

In [ ]:
# ── Configuración ──────────────────────────────────────────
N_COMPONENTES = 0.95  # ← CAMBIAR: número entero (ej: 5) o proporción de varianza a retener (ej: 0.95)
# ───────────────────────────────────────────────────────────

pca = PCA(n_components=N_COMPONENTES, random_state=42)
X_train_pca = pca.fit_transform(X_train[COLUMNAS_ESCALAR])
X_test_pca  = pca.transform(X_test[COLUMNAS_ESCALAR])

# Varianza explicada por cada componente
varianza_acum = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(varianza_acum) + 1), varianza_acum, marker='o')
plt.axhline(y=0.95, color='red', linestyle='--', label='95% varianza')
plt.xlabel('Número de componentes')
plt.ylabel('Varianza explicada acumulada')
plt.title('PCA — Varianza explicada acumulada')
plt.legend()
plt.tight_layout()
plt.show()

print(f'✅ Componentes generadas: {pca.n_components_}')
print(f'Varianza total explicada: {varianza_acum[-1]:.2%}')

---
# BLOQUE 10 — Pipeline de Sklearn (Avanzado)

Encadenar múltiples transformaciones en un objeto reutilizable y reproducible.  
**Cuándo usarlo:** en proyectos donde necesitás garantizar que el mismo preprocesamiento se aplique siempre en el mismo orden, especialmente en producción.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ── Configuración ──────────────────────────────────────────
COLS_NUMERICAS    = ['col_num_1', 'col_num_2']   # ← CAMBIAR
COLS_CATEGORICAS  = ['col_cat_1', 'col_cat_2']   # ← CAMBIAR
# ───────────────────────────────────────────────────────────

# Pipeline para variables numéricas: imputar nulos → escalar
pipe_numericas = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Pipeline para variables categóricas: imputar nulos → One-Hot
pipe_categoricas = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# Combinador: aplica cada pipeline a sus columnas correspondientes
preprocessor = ColumnTransformer([
    ('num', pipe_numericas,   COLS_NUMERICAS),
    ('cat', pipe_categoricas, COLS_CATEGORICAS)
])

# Aplicar sobre train y test
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

print(f'✅ Pipeline aplicado')
print(f'Train: {X_train_prep.shape} | Test: {X_test_prep.shape}')

---
# 📋 Referencia Rápida

| Bloque | Técnica | Nivel | Caso de uso típico |
|---|---|---|---|
| 1 | Variables derivadas | Básico | Diferencia de GPA, ratio de ingresos |
| 2 | Reglas con .apply() | Básico | Costos estimados, scoring personalizado |
| 3 | Binning | Básico-Intermedio | Segmentar edades, rangos de ingresos |
| 4 | Extracción de fechas | Intermedio | Estacionalidad, antigüedad |
| 5 | Encoding categórico | Intermedio | Preparación para modelos ML |
| 6 | Transformaciones log/sqrt | Intermedio | Corregir distribuciones sesgadas |
| 7 | Escalado | Intermedio | Regresión Ridge/Lasso, SVM, KNN |
| 8 | Selección de features | Avanzado | Reducir ruido, mejorar rendimiento |
| 9 | PCA | Avanzado | Comprimir variables correlacionadas |
| 10 | Pipeline sklearn | Avanzado | Producción, reproducibilidad |

---
*Template de Feature Engineering — ds-toolkit by andressonsino*